## Import

In [ ]:
import os
import sep
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from datetime import datetime
import matplotlib.dates as mdates
import matplotlib.cm as cm
import astroalign as aa
from pandas import read_csv

## 目标仿真

### 相关仿真函数

In [ ]:
def _gaussian_psf(radius_px: float, size: int = None) -> np.ndarray:
    sigma = max(0.2, float(radius_px))
    if size is None:
        size = int(np.ceil(6 * sigma)) | 1
    c = size // 2
    y, x = np.mgrid[-c:c+1, -c:c+1]
    psf = np.exp(-(x*x + y*y) / (2 * sigma * sigma))
    psf /= psf.sum()
    return psf.astype(np.float32)

def _motion_kernel(length_px: float, angle_rad: float, width: float = 1.0) -> np.ndarray:
    L = max(1.0, float(length_px))
    size = int(np.ceil(L)) | 1
    c = size // 2
    y, x = np.mgrid[-c:c+1, -c:c+1]
    ca, sa = np.cos(angle_rad), np.sin(angle_rad)
    u =  ca * x + sa * y
    v = -sa * x + ca * y
    line = (np.abs(u) <= (L / 2)).astype(np.float32)
    blur = np.exp(-(v * v) / (2 * (max(0.3, width) ** 2)))
    k = line * blur
    k /= k.sum()
    return k.astype(np.float32)

def _fft_convolve_full(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """full convolution (size = (Ha+Hb-1, Wa+Wb-1)) via FFT"""
    Ha, Wa = a.shape
    Hb, Wb = b.shape
    H, W = Ha + Hb - 1, Wa + Wb - 1
    fa = np.fft.rfft2(a, s=(H, W))
    fb = np.fft.rfft2(b, s=(H, W))
    out = np.fft.irfft2(fa * fb, s=(H, W))
    return out.astype(np.float32)

def _place_kernel_add(img: np.ndarray, ker: np.ndarray, y: float, x: float, gain: float) -> None:
    """越界直接丢弃（不环绕）。"""
    H, W = img.shape
    kh, kw = ker.shape
    cy, cx = kh // 2, kw // 2

    y0 = int(np.floor(y))
    x0 = int(np.floor(x))
    dy = y - y0
    dx = x - x0

    weights = [
        (y0,   x0,   (1-dy)*(1-dx)),
        (y0+1, x0,   dy*(1-dx)),
        (y0,   x0+1, (1-dy)*dx),
        (y0+1, x0+1, dy*dx),
    ]

    for yy, xx, w in weights:
        top = yy - cy
        left = xx - cx

        # 计算 ker 放到 img 上的重叠区域（img坐标系）
        r0 = max(0, top)
        c0 = max(0, left)
        r1 = min(H, top + kh)
        c1 = min(W, left + kw)

        # 如果完全不相交，直接跳过
        if r0 >= r1 or c0 >= c1:
            continue

        # 对应到 ker 的裁剪区域
        kr0 = r0 - top
        kc0 = c0 - left
        kr1 = kr0 + (r1 - r0)
        kc1 = kc0 + (c1 - c0)

        img[r0:r1, c0:c1] += (gain * w) * ker[kr0:kr1, kc0:kc1]

def simulate_trajectory_projection_with_interval(
    seed: int,
    img_shape=(256, 256),
    radius_px=1.5,
    peak=2000.0,
    start_xy=None,              # (x, y)
    speed_px_s=8.0,
    angle_deg=None,
    n_frames=50,
    exposure_s=0.01,
    frame_interval_s=0.04,      
    jitter_px=0.5,
    scintillation=0.15,
    background=50.0,
    shot_noise=False,
    read_noise=3.0,
):
    rng = np.random.default_rng(seed)
    H, W = img_shape

    if angle_deg is None:
        angle_deg = rng.uniform(0, 90)
    angle = np.deg2rad(float(angle_deg))
    vx = float(speed_px_s) * np.cos(angle)
    vy = float(speed_px_s) * np.sin(angle)

    if start_xy is None:
        x0 = rng.uniform(W//4, W//4*3)
        y0 = rng.uniform(H//4, H//4*3)
    else:
        x0, y0 = map(float, start_xy)

    # 每帧有效核：PSF * 帧内运动模糊（小核卷小核 -> full）
    psf = _gaussian_psf(radius_px)
    blur_len = max(0.0, float(speed_px_s) * float(exposure_s))
    if blur_len >= 1.0:
        mk = _motion_kernel(blur_len, angle, width=max(0.8, radius_px * 0.6))
        eff = _fft_convolve_full(psf, mk)
        eff /= eff.sum()  # 归一化（保持能量守恒）
    else:
        eff = psf

    tar = np.zeros((H, W), dtype=np.float32)

    for k in range(int(n_frames)):
        t = k * float(frame_interval_s)

        x = x0 + vx * t + rng.normal(0.0, jitter_px)
        y = y0 + vy * t + rng.normal(0.0, jitter_px)

        pk = float(peak) * (1.0 + rng.normal(0.0, scintillation))
        pk = max(0.0, pk)

        _place_kernel_add(tar, eff, y, x, pk)

    ratio = peak / (tar.max()+1e-6)
    tar = ratio * tar
    out = tar + float(background)
    if shot_noise:
        out = np.clip(out, 0, None)   
        out = rng.poisson(out).astype(np.float32)
    if read_noise > 0:
        out = out + rng.normal(0.0, float(read_noise), size=out.shape).astype(np.float32)

    return out, tar

def global_bbox_from_projection_image(img: np.ndarray, thr: float):
    m = img > thr
    ys, xs = np.where(m)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    return (x1, y1, x2, y2)

### 测试仿图效果

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']
save_dir = 'dataset/imgs/'
imgs = [x for path_dir in path_dirs for x in glob.glob(path_dir + '/*.tif')]


content = []
for i, path in enumerate(imgs[-3:]):
    img = cv2.imread(path, -1)[:1024, :1024]
    seed = i
    rng = np.random.default_rng(seed)
    exp = rng.normal(1, 0.3)
    img_pt, tar_pt = simulate_trajectory_projection_with_interval(
        seed=seed,
        img_shape = img.shape,
        n_frames=rng.integers(5, 30),
        speed_px_s=rng.integers(1, 5),
        exposure_s=exp,
        frame_interval_s=exp*rng.integers(2,15),  
        radius_px= max(rng.normal(2, 0.7), 0.5),
        peak= max(rng.normal(1, 0.5), 0.1) * img.std(),
        angle_deg=rng.uniform(-90, 90),
        background=0)

    seed += 1
    rng = np.random.default_rng(seed)
    img_line, tar_line = simulate_trajectory_projection_with_interval(
        seed=seed,
        img_shape = img.shape,
        n_frames=rng.integers(5, 30),
        speed_px_s=rng.integers(3, 23),
        exposure_s=exp,
        frame_interval_s=exp*rng.integers(2,5),  
        radius_px=rng.normal(1, 0.3),
        peak= max(rng.normal(0.5, 0.5), 0.1) * img.std(),
        angle_deg=rng.uniform(-90, 90),
        background=0)   
    
    oup = img + img_pt + img_line
    tmp = [i]
    for tar in [tar_line, tar_pt]:
        th = np.percentile(tar, 99.9)
        x1, y1, x2, y2 = global_bbox_from_projection_image(tar, th)
        tmp += [x1, y1, x2, y2]
    content.append([tmp])

    vmin, vmax = np.percentile(oup, (0.5, 99.5))
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(oup, cmap='gray', vmin=vmin, vmax=vmax)
    for i in range(2):
        x1, y1, x2, y2 = tmp[i*4+1:i*4+5]
        axes[0].add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=1, edgecolor='r', facecolor='none'))
    axes[1].imshow(img_line+img_pt, cmap='gray')
    for ax in axes: ax.axis('off')
    plt.show()

### 仿图生成代码

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']
save_dir = 'dataset/imgs/'
imgs = [x for path_dir in path_dirs for x in glob.glob(path_dir + '/*.tif')]

a = 1024
s = 512
idx = 1
content = []

for q, path in enumerate(tqdm(imgs)):
    ori = cv2.imread(path, -1).astype(np.float32)
    h, w = ori.shape
    nw = (w-a)//s+1
    nh = (h-a)//s+1
    for i in range(nh):
        for j in range(nw):
            x1, y1= j*s, i*s
            x2, y2 = x1+a, y1+a
            img = ori[y1:y2, x1:x2]
            try:
                seed = idx
                rng = np.random.default_rng(seed)
                exp = rng.normal(1, 0.3)
                img_pt, tar_pt = simulate_trajectory_projection_with_interval(
                    seed=seed,
                    img_shape = img.shape,
                    n_frames=rng.integers(5, 30),
                    speed_px_s=rng.integers(1, 5),
                    exposure_s=exp,
                    frame_interval_s=exp*rng.integers(2,15),  
                    radius_px= max(rng.normal(2, 0.7), 0.5),
                    peak= max(rng.normal(1, 0.5), 0.1) * img.std(),
                    angle_deg=rng.uniform(-90, 90),
                    background=0)

                seed += 1
                rng = np.random.default_rng(seed)
                img_line, tar_line = simulate_trajectory_projection_with_interval(
                    seed=seed,
                    img_shape = img.shape,
                    n_frames=rng.integers(5, 30),
                    speed_px_s=rng.integers(3, 23),
                    exposure_s=exp,
                    frame_interval_s=exp*rng.integers(2,5),  
                    radius_px=rng.normal(1, 0.3),
                    peak= max(rng.normal(0.5, 0.5), 0.1) * img.std(),
                    angle_deg=rng.uniform(-90, 90),
                    background=0)   
                
                oup = img + img_pt + img_line
                tmp = [idx]
                for tar in [tar_line, tar_pt]:
                    th = np.percentile(tar, 99.9)
                    if th > 0:
                        x1, y1, x2, y2 = global_bbox_from_projection_image(tar, th)
                        tmp += [x1, y1, x2, y2]
                    else:
                        tmp += [-1, -1,- 1, -1]
                content.append(tmp)
                
                path_save = f'{save_dir}/{idx:03d}.tif'
                if not cv2.imwrite(path_save, oup):
                    print(f'Failed to save {path_save}')
                idx += 1
            except:
                print(f'Failed to generate {idx}--{path}')

with open(f'dataset/labels.txt', 'w') as f:
    for line in content:
        f.write(f'{line[0]:03d},{line[1]},{line[2]},{line[3]},{line[4]},{line[5]},{line[6]},{line[7]},{line[8]}\n')

### 星点抑制后的仿图生成代码

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']
meds = []
for path_dir in path_dirs:
    aligns = []
    paths = glob.glob(path_dir+'*.tif')
    for path in tqdm(paths):
        aligns.append(cv2.imread(path, -1))
    med = np.median(np.array(aligns), axis=0)
    bkg = sep.Background(np.ascontiguousarray(med, np.float32))
    bkg_img = np.array(bkg)
    tars, ext_map = sep.extract(med-bkg_img, 5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
    meds.append(ext_map)

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']
save_dir = 'dataset/cleans/'
imgs = [x for path_dir in path_dirs for x in glob.glob(path_dir + '/*.tif')]

a = 1024
s = 512
idx = 1
content = []

for num_dir, path_dir in enumerate(tqdm(path_dirs)):
    mask = meds[num_dir] > 0
    paths = glob.glob(path_dir+'*.tif')
    for path in tqdm(paths):
        ori = cv2.imread(path, -1).astype(np.float32)
        h, w = ori.shape
        nw = (w-a)//s+1
        nh = (h-a)//s+1
        for i in range(nh):
            for j in range(nw):
                x1, y1= j*s, i*s
                x2, y2 = x1+a, y1+a
                img = ori[y1:y2, x1:x2]
                mask_roi = mask[y1:y2, x1:x2]
                img[mask_roi] = np.median(img)
                try:
                    seed = idx
                    rng = np.random.default_rng(seed)
                    exp = rng.normal(1, 0.3)
                    img_pt, tar_pt = simulate_trajectory_projection_with_interval(
                        seed=seed,
                        img_shape = img.shape,
                        n_frames=rng.integers(5, 30),
                        speed_px_s=rng.integers(1, 5),
                        exposure_s=exp,
                        frame_interval_s=exp*rng.integers(2,15),  
                        radius_px= max(rng.normal(2, 0.7), 0.5),
                        peak= max(rng.normal(0.1, 0.03), 0.001) * img.max(),
                        angle_deg=rng.uniform(-90, 90),
                        background=0)

                    seed += 1
                    rng = np.random.default_rng(seed)
                    img_line, tar_line = simulate_trajectory_projection_with_interval(
                        seed=seed,
                        img_shape = img.shape,
                        n_frames=rng.integers(5, 30),
                        speed_px_s=rng.integers(3, 23),
                        exposure_s=exp,
                        frame_interval_s=exp*rng.integers(2,5),  
                        radius_px=rng.normal(1, 0.3),
                        peak= max(rng.normal(0.1, 0.03), 0.001) * img.max(),
                        angle_deg=rng.uniform(-90, 90),
                        background=0)   
                    
                    oup = img + img_pt + img_line
                    tmp = [idx]
                    for tar in [tar_line, tar_pt]:
                        th = np.percentile(tar, 99.9)
                        if th > 0:
                            x1, y1, x2, y2 = global_bbox_from_projection_image(tar, th)
                            tmp += [x1, y1, x2, y2]
                        else:
                            tmp += [-1, -1,- 1, -1]
                    content.append(tmp)
                    
                    path_save = f'{save_dir}/{idx:03d}.tif'     
                    if not cv2.imwrite(path_save, oup):
                        print(f'Failed to save {path_save}')
                    idx += 1
                except:
                    print(f'Failed to generate {idx}--{path}')  

with open(f'dataset/labels.txt', 'w') as f:
    for line in content:
        f.write(f'{line[0]:03d},{line[1]},{line[2]},{line[3]},{line[4]},{line[5]},{line[6]},{line[7]},{line[8]}\n')

### 转为YOLO训练所需格式

In [ ]:
# Build YOLO dataset from dataset/imgs + dataset/labels.txt
from pathlib import Path
import shutil
import random
import cv2
import numpy as np

# ===== configurable =====
imgs_dir = Path('dataset/cleans')
labels_file = Path('dataset/labels.txt')
out_root = Path('dataset/yolo_det_clean')
train_ratio = 0.8  # 4:1 split
seed = 42
class_names = ['line']  # class 0
clean_output = True
RATIO = (0.01, 99.99) # (0.5, 99.5)
# ========================

assert imgs_dir.exists(), f'Missing image dir: {imgs_dir}'
assert labels_file.exists(), f'Missing labels file: {labels_file}'

if clean_output and out_root.exists():
    shutil.rmtree(out_root)

# Collect images
exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
images = sorted([p for p in imgs_dir.iterdir() if p.is_file() and p.suffix.lower() in exts])
assert images, f'No images found in {imgs_dir}'

# Parse labels into dict by idx (critical: do NOT align by file order)
# format: idx,x1,y1,x2,y2,x3,y3,x4,y4
labels_by_idx = {}
with labels_file.open('r', encoding='utf-8') as f:
    for ln, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        parts = [x.strip() for x in line.split(',')]
        if len(parts) != 9:
            raise ValueError(f'Bad label format at line {ln}: expect 9 fields, got {len(parts)}')

        idx = int(parts[0])
        vals = list(map(float, parts[1:]))
        box1 = tuple(vals[:4])
        box2 = tuple(vals[4:8])
        if idx in labels_by_idx:
            raise ValueError(f'Duplicate idx in labels.txt: {idx} (line {ln})')
        labels_by_idx[idx] = (box1, box2)

# Pair by image stem idx (e.g. 007.tif -> idx 7)
paired = []
missing_label = []
for img_path in images:
    try:
        img_idx = int(img_path.stem)
    except ValueError:
        raise ValueError(f'Image name is not numeric stem: {img_path.name}')

    if img_idx not in labels_by_idx:
        missing_label.append(img_path.name)
        continue
    paired.append((img_path, labels_by_idx[img_idx]))

if missing_label:
    raise ValueError(f'Missing label for {len(missing_label)} images, e.g. {missing_label[:5]}')

# Detect labels without image
img_idx_set = {int(p.stem) for p in images}
orphan_labels = sorted(set(labels_by_idx) - img_idx_set)
if orphan_labels:
    raise ValueError(f'labels.txt has {len(orphan_labels)} idx without image, e.g. {orphan_labels[:5]}')

# Make output dirs
for split in ['train', 'val']:
    (out_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (out_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Shuffle then split
rng = random.Random(seed)
# rng.shuffle(paired)
train_count = int(len(paired) * train_ratio)
train_set = paired[:train_count]
val_set = paired[train_count:]

def trunc_img(img, ratio=(0.5, 99.5)):
    vmin, vmax = np.percentile(img, ratio)
    img = np.clip(img, vmin, vmax)
    out = (img - vmin) / (vmax - vmin + 1e-8) * 255
    return out.astype(np.uint8)

def convert_to_yolo_xyxy(x1, y1, x2, y2, w, h):
    # clip to image bounds in pixel space, then normalize
    x1 = max(0.0, min(float(w), x1))
    x2 = max(0.0, min(float(w), x2))
    y1 = max(0.0, min(float(h), y1))
    y2 = max(0.0, min(float(h), y2))

    xa, xb = sorted((x1, x2))
    ya, yb = sorted((y1, y2))
    bw = xb - xa
    bh = yb - ya
    if bw <= 0 or bh <= 0:
        return None

    xc = (xa + xb) / 2.0 / w
    yc = (ya + yb) / 2.0 / h
    bw = bw / w
    bh = bh / h
    return (xc, yc, bw, bh)

def write_split(pairs, split):
    for img_path, (box1, box2) in tqdm(pairs):
        img = cv2.imread(str(img_path), -1)
        if img is None:
            raise ValueError(f'Failed to read image: {img_path}')
        h, w = img.shape[:2]

        # save 8-bit training image
        dst_img = out_root / 'images' / split / f'{img_path.stem}.png'
        cv2.imwrite(str(dst_img), trunc_img(img, RATIO))

        # write labels
        yolo_lines = []
        for (x1, y1, x2, y2) in (box1, box2):
            if x1 == -1 or y1 == -1:
                continue
            y = convert_to_yolo_xyxy(x1, y1, x2, y2, w=w, h=h)
            if y is None:
                continue
            yolo_lines.append(f'0 {y[0]:.6f} {y[1]:.6f} {y[2]:.6f} {y[3]:.6f}')

        # remove exact duplicates while preserving order
        dedup = []
        seen = set()
        for line in yolo_lines:
            if line not in seen:
                dedup.append(line)
                seen.add(line)

        dst_lbl = out_root / 'labels' / split / f'{img_path.stem}.txt'
        with dst_lbl.open('w', encoding='utf-8') as f:
            if dedup:
                f.write('\n'.join(dedup) + '\n')

write_split(train_set, 'train')
write_split(val_set, 'val')

yaml_text = (
    f'path: {out_root.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n'
    'channels: 1\n'
    'names:\n'
    f'  0: {class_names[0]}\n'
)
(out_root / 'data.yaml').write_text(yaml_text, encoding='utf-8')

print(f'Total paired images: {len(paired)}')
print(f'Train: {len(train_set)} | Val: {len(val_set)}')
print(f'Dataset ready at: {out_root}')
print(f'Config: {out_root / "data.yaml"}')


### 测试仿图效果

In [ ]:
idx = 000
sample = cv2.imread(f'imgs/generate/{idx:03d}.tif', -1)
with open('imgs/generate/labels.txt', 'r') as f:
    coords = np.array([f.split('\n')[0].split(',') for f in f.readlines()], np.int64)

i, x1, y1, x2, y2 = coords[idx-1]
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
vmin, vmax = np.percentile(sample, (0.5, 99.5))
ax.imshow(sample, cmap='gray', vmin=vmin, vmax=vmax)
ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=1, edgecolor='r', facecolor='none'))
ax.axis('off')
plt.show()

## 静态背景真实数据

In [ ]:
def load_tj_img(path, denoise=True):
    img = cv2.imread(path, -1).astype(np.float32)
    if not denoise:
        return img
    med = cv2.medianBlur(img, 3)
    sub = img - med
    mask = (sub > med) & (med < med.mean()+med.std())
    oup = img.copy()
    oup[mask] = med[mask]
    return oup

In [ ]:
a = 2048
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']

### 导出测试数据（原始对齐图像）

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0: continue
    aligns = []
    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        aligns.append(img2)
    aligns = np.array(aligns)
    oup = np.max(aligns, axis=0)
    med = np.median(aligns, axis=0)

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a
            roi = oup[y1:y2, x1:x2]
            
            vmin, vmax = np.percentile(roi, (0.5, 99.5))
            roi = (roi - vmin) / (vmax - vmin)
            roi = np.clip(roi, 0, 1)
            roi = (roi * 255).astype(np.uint8)
            if not cv2.imwrite(f'dataset/test/still_{idx:03d}.png', roi):
                print(f'Failed to save {idx}')
            idx += 1

### 导出测试数据（删除星点）

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0: continue
    aligns = []
    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        aligns.append(img2)
    aligns = np.array(aligns)
    oup = np.max(aligns, axis=0)
    med = np.median(aligns, axis=0)

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a
            med_roi = med[y1:y2, x1:x2]
            bkg = sep.Background(np.ascontiguousarray(med_roi, np.float32))
            bkg_img = np.array(bkg)
            tars, ext_map = sep.extract(med_roi - bkg_img, 1.5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
            star_mask_roi = ext_map > 0
            roi = oup[y1:y2, x1:x2]

            val = np.percentile(roi, 1)
            roi[star_mask_roi] = val
            vmin, vmax = np.percentile(roi, (0.01, 99.99))
            roi = (roi - vmin) / (vmax - vmin)
            roi = np.clip(roi, 0, 1)
            roi = (roi * 255).astype(np.uint8)
            if not cv2.imwrite(f'dataset/test_clean/still_{idx:03d}.png', roi):
                print(f'Failed to save {idx}')
            idx += 1

### 测试数据

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ2/decode/']

path_dir = path_dirs[0]
imgs = glob.glob(path_dir + '/*.tif')
aligns = []
for path in imgs: 
    img2 = cv2.imread(path, -1)
    aligns.append(img2)
aligns = np.array(aligns)

In [ ]:
med = np.median(aligns, axis=0)
maxi = np.max(aligns, axis=0)
bkg = sep.Background(np.ascontiguousarray(med, np.float32))
bkg_img = np.array(bkg)
tars, ext_map = sep.extract(med-bkg_img, 5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
sub = maxi - med
bkg_val = bkg.globalback
sub[ext_map>0] = sub.mean()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
vmin, vmax = np.percentile(sub, (0.5, 99.5))
axes[0].imshow(sub, cmap='gray', vmin=vmin, vmax=vmax)
axes[1].imshow(ext_map>0)
for ax in axes: ax.axis('off')
plt.show()

In [ ]:
bkg_tmp = sep.Background(np.ascontiguousarray(sub))
_, ext_tmp = sep.extract(sub-bkg_tmp, 1.5, err=bkg_tmp.globalrms, deblend_cont=1, segmentation_map=True)
plt.figure(figsize=(8, 8))
plt.imshow(ext_tmp>0)
plt.axis('off')

## 动态背景真实数据

In [ ]:
a = 2048
path_dirs = ['/mnt/e/Imgs/04-RWZ/20220907/822070_2000/decode/',
            '/mnt/e/Imgs/03-TJStars/150ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/200ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/5000ms/decode/',
            '/mnt/e/Imgs/03-TJStars/TJ1/decode/',]

### 导出测试数据（原始对齐图像）

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:1], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0:
        continue
    img1 = load_tj_img(imgs[0])
    aligns = [img1]
    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        img21, footprint = aa.register(img2, img1)
        aligns.append(img21)
    aligns = np.array(aligns)
    oup = np.max(aligns, axis=0)
    med = np.median(aligns, axis=0)

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a
            roi = oup[y1:y2, x1:x2]

            vmin, vmax = np.percentile(roi, (0.5, 99.5))
            roi = (roi - vmin) / (vmax - vmin)
            roi = np.clip(roi, 0, 1)
            roi = (roi * 255).astype(np.uint8)
            if not cv2.imwrite(f'dataset/test/{idx:03d}.png', roi):
                print(f'Failed to save {idx}')
            idx += 1

### 导出测试数据(删除星点)

In [ ]:
idx = 1
for path_dir in tqdm(path_dirs[:1], desc='Current Directory'):
    imgs = sorted(glob.glob(path_dir + '/*.tif'))
    if len(imgs) == 0:
        continue
    img1 = load_tj_img(imgs[0])
    aligns = [img1]
    valid_masks = [np.ones_like(img1, dtype=bool)]

    for path in tqdm(imgs[1:], desc='Generate Mask'):
        img2 = load_tj_img(path)
        img21, footprint = aa.register(img2, img1)
        aligns.append(img21)
        valid_masks.append(footprint)
    aligns = np.array(aligns)
    overlap_mask_all = np.logical_and.reduce(np.stack(valid_masks, axis=0), axis=0)
    oup = np.max(aligns, axis=0)
    med = np.median(aligns, axis=0)

    h, w = oup.shape
    nh, nw = h // a, w // a
    for i in range(nh):
        for j in range(nw):
            y1, y2 = i * a, (i + 1) * a
            x1, x2 = j * a, (j + 1) * a

            med_roi = med[y1:y2, x1:x2]
            bkg = sep.Background(np.ascontiguousarray(med_roi, np.float32))
            bkg_img = np.array(bkg)
            tars, ext_map = sep.extract(med_roi - bkg_img, 1.5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
            star_mask_roi = ext_map > 0

            roi = oup[y1:y2, x1:x2]
            overlap_roi = overlap_mask_all[y1:y2, x1:x2]
            val = np.percentile(roi, 1)
            roi[star_mask_roi] = val
            roi[overlap_roi] = roi.min()
            vmin, vmax = np.percentile(roi, (0.01, 99.99))
            roi = (roi - vmin) / (vmax - vmin)
            roi = np.clip(roi, 0, 1)
            roi = (roi * 255).astype(np.uint8)

            if not cv2.imwrite(f'dataset/test_clean/{idx:03d}.png', roi):
                print(f'Failed to save {idx}')
            idx += 1

### 测试数据

In [ ]:
path_dirs = ['/mnt/e/Imgs/03-TJStars/TJ1/decode/',
            '/mnt/e/Imgs/03-TJStars/150ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/200ms-4k4k-motion/decode/',
            '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/']

path_dir = path_dirs[3]
imgs = glob.glob(path_dir + '/*.tif')
img1 = cv2.imread(imgs[0], -1)
aligns = [img1]
for path in imgs[1:]: 
    img2 = cv2.imread(path, -1)
    img21, _ = aa.register(img2, img1)
    aligns.append(img21)
aligns = np.array(aligns)
maxi = np.max(aligns, axis=0)
med = np.median(np.array(aligns), axis=0)

In [ ]:
bkg = sep.Background(np.ascontiguousarray(med, np.float32))
bkg_img = np.array(bkg)
tars, ext_map = sep.extract(med-bkg_img, 3, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
mask = (ext_map>0).astype(np.uint8)
oup = np.asarray(maxi, np.float32)
bkg_oup = sep.Background(np.ascontiguousarray(oup, np.float32))
oup = oup - np.array(bkg_oup)
oup[mask>0] = np.median(oup)

In [ ]:
tmp = oup[:2048, 2048:]
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
vmin, vmax = np.percentile(tmp, (0.1, 99.9))
ax.imshow(tmp, cmap='gray', vmin=vmin, vmax=vmax)
ax.axis('off')
plt.show()

## OBB数据合成

In [ ]:
# OBB simulation module (based on current star-suppressed synthetic pipeline)
from pathlib import Path
import glob
import cv2
import numpy as np
from tqdm import tqdm


def order_quad_clockwise(pts):
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    c = pts.mean(axis=0)
    ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
    pts = pts[np.argsort(ang)]
    i0 = np.argmin(pts[:, 0] + pts[:, 1])
    pts = np.roll(pts, -i0, axis=0)
    return pts


def global_obb_from_projection_image(img: np.ndarray, thr: float, min_pixels: int = 4):
    """Build a stable OBB from thresholded target via geometric PCA after 5x5 dilation."""
    if img is None or img.size == 0 or not np.isfinite(thr):
        return None

    # 1) threshold + 5x5 dilation as requested
    mask = (img > thr).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    ys, xs = np.where(mask > 0)
    if len(xs) < min_pixels:
        return None

    # 2) point cloud in image coords (x, y)
    pts = np.stack([xs, ys], axis=1).astype(np.float32)

    # 3) principal axes (geometric orientation)
    center = pts.mean(axis=0)
    pts0 = pts - center
    cov = np.cov(pts0, rowvar=False)
    if not np.all(np.isfinite(cov)):
        return None

    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    v1 = eigvecs[:, order[0]].astype(np.float32)
    v2 = eigvecs[:, order[1]].astype(np.float32)

    # make axis pair right-handed for stable corner ordering
    if np.cross(v1, v2) < 0:
        v2 = -v2

    # 4) tight rectangle in PCA frame
    p1 = pts0 @ v1
    p2 = pts0 @ v2
    min1, max1 = float(p1.min()), float(p1.max())
    min2, max2 = float(p2.min()), float(p2.max())

    if (max1 - min1) < 1e-6 or (max2 - min2) < 1e-6:
        return None

    # corners in (v1, v2) coordinates: [tl, tr, br, bl] before final ordering
    local = np.array([
        [min1, min2],
        [max1, min2],
        [max1, max2],
        [min1, max2],
    ], dtype=np.float32)

    corners = center[None, :] + local[:, :1] * v1[None, :] + local[:, 1:] * v2[None, :]
    return order_quad_clockwise(corners)

# ===== configurable =====
path_dirs = [
    '/mnt/e/Imgs/03-TJStars/150ms-2k2k/decode/',
    '/mnt/e/Imgs/03-TJStars/3000ms-4k4k/decode/',
    '/mnt/e/Imgs/03-TJStars/TJ2/decode/'
]
out_img_dir = Path('dataset/obb_sim/imgs')
out_label_file = Path('dataset/obb_sim/labels_obb.txt')
patch_size = 1024
stride = 512
start_idx = 1
# ========================

out_img_dir.mkdir(parents=True, exist_ok=True)

idx = start_idx
content = []

meds = []
for path_dir in path_dirs:
    aligns = []
    paths = glob.glob(path_dir+'*.tif')
    for path in tqdm(paths, desc='Generate Aigned Img'):
        aligns.append(cv2.imread(path, -1))
    med = np.median(np.array(aligns), axis=0)
    bkg = sep.Background(np.ascontiguousarray(med, np.float32))
    bkg_img = np.array(bkg)
    tars, ext_map = sep.extract(med-bkg_img, 5, err=bkg.globalrms, deblend_cont=1, segmentation_map=True)
    meds.append(ext_map)

for num_dir, path_dir in enumerate(tqdm(path_dirs, desc='Generate Destar Img')):
    mask = meds[num_dir] > 0
    paths = glob.glob(path_dir+'*.tif')
    for path in paths:
        ori = cv2.imread(path, -1)
        if ori is None:
            continue
        ori = ori.astype(np.float32)
        h, w = ori.shape
        nw = (w - patch_size) // stride + 1
        nh = (h - patch_size) // stride + 1

        for i in range(nh):
            for j in range(nw):
                x1, y1 = j * stride, i * stride
                x2, y2 = x1 + patch_size, y1 + patch_size
                img = ori[y1:y2, x1:x2]
                mask_roi = mask[y1:y2, x1:x2]
                img[mask_roi] = np.median(img)
                try:
                    seed = idx
                    rng = np.random.default_rng(seed)
                    exp = rng.normal(1, 0.3)
                    img_pt, tar_pt = simulate_trajectory_projection_with_interval(
                        seed=seed,
                        img_shape=img.shape,
                        n_frames=rng.integers(5, 30),
                        speed_px_s=rng.integers(1, 5),
                        exposure_s=exp,
                        frame_interval_s=exp * rng.integers(2, 15),
                        radius_px=max(rng.normal(2, 0.7), 0.5),
                        peak= max(rng.normal(0.1, 0.03), 0.001) * img.max(),
                        angle_deg=rng.uniform(-90, 90),
                        background=0,
                    )

                    seed += 1
                    rng = np.random.default_rng(seed)
                    img_line, tar_line = simulate_trajectory_projection_with_interval(
                        seed=seed,
                        img_shape=img.shape,
                        n_frames=rng.integers(5, 30),
                        speed_px_s=rng.integers(3, 23),
                        exposure_s=exp,
                        frame_interval_s=exp * rng.integers(2, 5),
                        radius_px=max(rng.normal(1, 0.3), 0.2),
                        peak= max(rng.normal(0.1, 0.03), 0.001) * img.max(),
                        angle_deg=rng.uniform(-90, 90),
                        background=0,
                    )

                    syn = img + img_pt + img_line
                    row = [idx]
                    for tar in [tar_line, tar_pt]:
                        th = np.percentile(tar, 99.9)
                        obb = global_obb_from_projection_image(tar, th)
                        if obb is None:
                            row += [-1.0] * 8
                        else:
                            row += obb.reshape(-1).tolist()
                    content.append(row)
                    cv2.imwrite(str(out_img_dir / f'{idx:03d}.tif'), syn)
                    idx += 1
                except Exception as e:
                    print(f'Failed idx={idx}: {e}')

with out_label_file.open('w', encoding='utf-8') as f:
    for row in content:
        vals = ','.join([f'{row[0]:03d}'] + [f'{v:.6f}' for v in row[1:]])
        f.write(vals + '\n')

print(f'OBB sim done. images={len(content)}, labels={out_label_file}')


转为yolo-obb格式

In [ ]:
# Convert OBB simulation output to YOLO-OBB dataset format
from pathlib import Path
import random
import shutil
import cv2
import numpy as np


def order_quad_clockwise(pts):
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    c = pts.mean(axis=0)
    ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
    pts = pts[np.argsort(ang)]
    i0 = np.argmin(pts[:, 0] + pts[:, 1])
    pts = np.roll(pts, -i0, axis=0)
    return pts


def polygon_area(pts):
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def trunc_img(img, ratio=(0.5, 99.5)):
    vmin, vmax = np.percentile(img, ratio)
    img = np.clip(img, vmin, vmax)
    out = (img - vmin) / (vmax - vmin + 1e-8) * 255
    return out.astype(np.uint8)


# ===== configurable =====
imgs_dir = Path('dataset/obb_sim/imgs')
labels_file = Path('dataset/obb_sim/labels_obb.txt')
out_root = Path('dataset/yolo_obb_auto')
train_ratio = 0.8
seed = 42
class_names = ['line']
clean_output = True
RATIO = (0.01, 99.99)
# ========================

assert imgs_dir.exists(), f'Missing image dir: {imgs_dir}'
assert labels_file.exists(), f'Missing labels file: {labels_file}'

if clean_output and out_root.exists():
    shutil.rmtree(out_root)

exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
images = sorted([p for p in imgs_dir.iterdir() if p.is_file() and p.suffix.lower() in exts])
assert images, f'No images found in {imgs_dir}'

labels_by_idx = {}
with labels_file.open('r', encoding='utf-8') as f:
    for ln, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        parts = [x.strip() for x in line.split(',')]
        if len(parts) != 17:
            raise ValueError(f'Bad OBB label format at line {ln}: expect 17 fields, got {len(parts)}')
        idx = int(parts[0])
        vals = list(map(float, parts[1:]))
        labels_by_idx[idx] = (vals[:8], vals[8:16])

paired = []
for img_path in images:
    img_idx = int(img_path.stem)
    if img_idx not in labels_by_idx:
        raise ValueError(f'Missing label for image: {img_path.name}')
    paired.append((img_path, labels_by_idx[img_idx]))

for split in ['train', 'val']:
    (out_root / 'images' / split).mkdir(parents=True, exist_ok=True)
    (out_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

rng = random.Random(seed)
rng.shuffle(paired)
train_count = int(len(paired) * train_ratio)
train_set = paired[:train_count]
val_set = paired[train_count:]


def to_yolo_obb_line(obb, w, h):
    if obb[0] < 0:
        return None
    pts = np.asarray(obb, dtype=np.float32).reshape(4, 2)
    pts[:, 0] = np.clip(pts[:, 0], 0, w)
    pts[:, 1] = np.clip(pts[:, 1], 0, h)
    pts = order_quad_clockwise(pts)
    if polygon_area(pts) < 2.0:
        return None
    pts[:, 0] /= float(w)
    pts[:, 1] /= float(h)
    vals = np.clip(pts.reshape(-1), 0.0, 1.0)
    return '0 ' + ' '.join([f'{v:.6f}' for v in vals])


def write_split(pairs, split):
    for img_path, (obb1, obb2) in pairs:
        img = cv2.imread(str(img_path), -1)
        if img is None:
            raise ValueError(f'Failed to read image: {img_path}')
        h, w = img.shape[:2]

        img8 = trunc_img(img, RATIO)
        img3 = cv2.cvtColor(img8, cv2.COLOR_GRAY2BGR)
        dst_img = out_root / 'images' / split / f'{img_path.stem}.png'
        cv2.imwrite(str(dst_img), img3)

        lines = []
        for obb in (obb1, obb2):
            line = to_yolo_obb_line(obb, w=w, h=h)
            if line is not None:
                lines.append(line)

        dedup = []
        seen = set()
        for line in lines:
            if line not in seen:
                dedup.append(line)
                seen.add(line)

        dst_lbl = out_root / 'labels' / split / f'{img_path.stem}.txt'
        with dst_lbl.open('w', encoding='utf-8') as f:
            if dedup:
                f.write('\n'.join(dedup) + '\n')

write_split(train_set, 'train')
write_split(val_set, 'val')

yaml_text = (
    f'path: {out_root.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n'
    'names:\n'
    f'  0: {class_names[0]}\n'
)
(out_root / 'data.yaml').write_text(yaml_text, encoding='utf-8')

print(f'Total paired images: {len(paired)}')
print(f'Train: {len(train_set)} | Val: {len(val_set)}')
print(f'YOLO-OBB dataset ready at: {out_root}')


校验obb准确性

In [ ]:
idx = 8
img = cv2.imread(f'dataset/yolo_obb_auto/images/train/{idx:03d}.png')
label = f'dataset/yolo_obb_auto/labels/train/{idx:03d}.txt'
with open(label) as f:
    lines = f.readlines()

fig, ax = plt.subplots(1, 1, figsize=(8,8))
plt.imshow(img)
for line in lines:
    pts = np.array(line.split('\n')[0].split()[1:], np.float32).reshape(-1, 2)*1024
    pts = np.vstack([pts, pts[0]])
    ax.plot(pts[:, 0], pts[:, 1])
ax.axis('off')

## 模型测试

单张图片

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/obb/yolo11n_obb_clean2/weights/best.pt")  # pretrained YOLO26n model

# Run batched inference on a list of images
results = model(["test_img.png"], imgsz=1024, conf=0.25)  # return a list of Results objects

# Process results list
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen
    result.save(filename="result.jpg")  # save to disk

### 测试原图

In [ ]:
direc = 'runs/obb/yolo11n_obb_clean'
test_dir = 'dataset/test'
conf = 0.25
model = YOLO(f"{direc}/weights/best.pt")

img_paths = sorted(glob.glob(f'{test_dir}/*.png'))
test_files = [cv2.imread(x) for x in img_paths]
test_files = [x for x in test_files if x is not None]
results = model(test_files, imgsz=test_files[0].shape[0], conf=conf)

n = len(results)
n_rows, n_cols = int(n/2+ 0.5), 2 
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
axes = [axes] if (n_rows == 1 and n_cols == 1) else axes.ravel()
for i, result in enumerate(results):
    im = result.plot(labels=False)
    axes[i].imshow(im[:, :, ::-1])  
    axes[i].axis('off')
for j in range(n, len(axes)): axes[j].axis('off') 
plt.tight_layout()
plt.show()

### 测试星点抑制图

In [ ]:
from ultralytics import YOLO
direc = 'runs/obb/yolo11n_obb_clean'
test_dir = 'dataset/test_clean'
conf = 0.25
model = YOLO(f"{direc}/weights/last.pt")

img_paths = sorted(glob.glob(f'{test_dir}/*.png'))
test_files = [cv2.imread(x) for x in img_paths]
test_files = [x for x in test_files if x is not None]
results = model(test_files, imgsz=test_files[0].shape[0], conf=conf)

n = len(results)
n_rows, n_cols = int(n/2+ 0.5), 2 
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
axes = [axes] if (n_rows == 1 and n_cols == 1) else axes.ravel()
for i, result in enumerate(results):
    im = result.plot(labels=False)
    axes[i].imshow(im[:, :, ::-1])  
    axes[i].axis('off')
for j in range(n, len(axes)): axes[j].axis('off') 
plt.tight_layout()
plt.show()